<b>Fonts</b>

https://sagemaker.readthedocs.io/en/stable/frameworks/sklearn/using_sklearn.html#deploy-endpoints-from-model-data

https://sagemaker.readthedocs.io/en/stable/frameworks/sklearn/sagemaker.sklearn.html

https://aws.amazon.com/blogs/machine-learning/preprocess-input-data-before-making-predictions-using-amazon-sagemaker-inference-pipelines-and-scikit-learn/

In [ ]:
!pip install pmdarima==2.0.4  # avoid errors

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import boto3
import sagemaker
import pmdarima as pm
from sagemaker.session import Session
from sagemaker.sklearn.estimator import SKLearn 
from sagemaker.sklearn.model import SKLearnModel 

from sagemaker.base_serializers import NumpySerializer, CSVSerializer, Base
from sagemaker.base_deserializers import NumpyDeserializer

import joblib
from time import strftime, gmtime 

In [ ]:
sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
sm_boto3 = boto3.client('sagemaker')
s3 = boto3.client('s3')

In [ ]:
data = pd.read_csv('CL_target.csv')
data.head(3)

# Split

In [ ]:
# Test is also Validation 
out_of_sample_size = 5
test = data.iloc[-out_of_sample_size:]
test = np.array(test['ChRec'])
data.tail(out_of_sample_size + 2)

In [ ]:
def forecast_accuracy(forecast, actual):
    ''' change this '''
    forecast, actual = np.array(forecast), np.array(actual)
    mape = np.mean(np.abs(forecast - actual)/np.abs(actual))*100  # MAPE
    mae = np.mean(np.abs(forecast - actual))    # MAE
    rmse = np.mean((forecast - actual)**2)**.5  # RMSE
    return({'mape(%)':mape, 'mae': mae, 'rmse':rmse})

In [ ]:
model = pm.auto_arima(
    data['ChRec'], suppress_warnings=True, seasonal=True, m=12, 
    information_criterion='oob', out_of_sample_size=out_of_sample_size
)
model

In [ ]:
forecast = model.predict(n_periods=out_of_sample_size)
for step in range(1, out_of_sample_size + 1):
    print("\n[INFO] Model Accuracy for Step", step)
    results = forecast_accuracy(forecast[:step], test[:step])
    for metric, value in results.items():
        print(f'\t {metric}: {value}')

In [ ]:
%%writefile tunning.py

import numpy as np
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
import joblib
import argparse
import os
import pmdarima as pm


def forecast_accuracy(forecast, actual):
    ''' change this '''
    #forecast, actual = np.asarray(forecast), np.asarray(actual)
    forecast, actual = np.array(forecast), np.array(actual)
    mape = np.mean(np.abs(forecast - actual)/np.abs(actual))*100  # MAPE
    mae = np.mean(np.abs(forecast - actual))    # MAE
    rmse = np.mean((forecast - actual)**2)**.5  # RMSE
    return({'mape(%)':mape, 'mae': mae, 'rmse':rmse})

def model_fn(model_dir):
    model = joblib.load(os.path.join(model_dir, "model.joblib"))
    return model

def predict_fn(steps, model):
    """
    param input_data: returned by the input_fn
    param model: returned by the model_fn
    returns: predictions 
    """
    forecast = model.predict(n_periods=int(steps))
    print("predict_fn-----------------------")
    print(steps, steps.dtype)
    return np.array(forecast)


if __name__=='__main__':
    parser = argparse.ArgumentParser()

    # where to write trained model artifacts
    parser.add_argument(
        "--model-dir", type=str, default=os.environ.get("SM_MODEL_DIR")
    )

    # Train and Test Data
    parser.add_argument("--data", type=str, default="CL_target.csv")
    
    # Validation and Testing 
    parser.add_argument("--out_of_sample_size", type=int, default=3)

    args, _ = parser.parse_known_args()

    print("\n[INFO] Reading data\n")
    data = pd.read_csv(args.data, columns='ChRec') # Only need this column
    test = np.array(data.iloc[-args.max_steps:])
    
    print("\n[INFO] Training data with ARIMA")
    # Seasonal = Month 
    model = pm.auto_arima(
        data, suppress_warnings=True, seasonal=True, m=12,
        information_criterion='oob', out_of_sample_size=args.out_of_sample_size
    )
    Print('Best Mode Params', model)
    

    model_path = os.path.join(args.model_dir, "model.joblib")
    print("\n[INFO] Saving best model at", model_path)
    joblib.dump(model, model_path)


    print("\n[INFO] Predicting data")
    forecast = model.predict(n_periods=out_of_sample_size)
    
    for step in range(1, args.out_of_sample_size + 1):
        print("\n[INFO] Model Accuracy for Step", step)
        results = forecast_accuracy(forecast[:step], test[:step])
        for metric, value in results.items():
            print(f'\t {metric}: {value}')
            
            


In [ ]:
# for debug porpuses only, csv data can be passed as an dependency 
out_of_sample_size =  3
sklearn_estimator = SKLearn(
    entry_point='tunning.py',
    instance_type='ml.m5.4xlarge',
    framework_version='1.2-1',
    base_job_name="autoArima-tunning",
    role=role,
    dependencies=['requirements.txt', 'CL_target.csv' ],
    hyperparams={"out_of_sample_size": out_of_sample_size},
    content_type="text/csv",
    accept="text/csv"
)

In [ ]:
# training the model with S3 data 
sklearn_estimator.fit() 

# A) Deploy estimator from instance

In [ ]:
endpoint_name = "fidelidade-" + strftime("%Y-%m-%d-%H-%M-%S", gmtime())
print("EndpointName deploy B) ={}".format(endpoint_name))

predictor = sklearn_estimator.deploy(
    initial_instance_count=1,
    instance_type='ml.m5.4xlarge',
    endpoint_name=endpoint_name,
    deserializer=NumpyDeserializer()
)

In [ ]:
predictor.__dict__

In [ ]:
forecast = predictor.predict(steps)
for step in range(1, out_of_sample_size):
    print('Accuracy Step', step)
    print(forecast_accuracy(forecast[:step], test[:step])

    

In [ ]:
sm_boto3.delete_endpoint(EndpointName=endpoint_name)